# 07 -- LSTM

Архитектура из одного из предыдущих коммитов, скип перебора гиперпараметров, теперь отрабатывает за $3$ часа вместо $20$!!!

- `AdamW`, `lr=5e-4`, `weight_decay=5e-4`;
- classifier -- 3 эпохи;
- regressor -- 13 эпох;
- positive threshold -- только `0`;
- `temperature=1`, `gamma=1`, без hard threshold.

$\textcolor{red}{\text{\#TODO таки число эпох еще увеличу + идею трешхолда я не брошу, см. git log}}$

Данные беру из [06_LSTM_Data_Preparation.ipynb](./06_LSTM_Data_Preparation.ipynb), static-признаки -- из [05_Data-Modeling.ipynb](./05_Data-Modeling.ipynb).

Использую две отдельные hybrid-модели с одинаковой архитектурой и разными весами:

1. **classifier** предсказывает $P(GMV > 0)$;
2. **regressor** учится только на $GMV > 0$ и предсказывает $\log(1 + GMV)$.

Финальный прогноз считаю сразу в log-space:

$$
\log(1 + \hat y) = P(GMV > 0) \cdot \widehat{\log(1 + GMV \mid GMV > 0)}.
$$

Для проверки оставляю последний размеченный cutoff как holdout _(чтобы заранее увидеть просадку по метрикам, как в предыдущем коммите, и прервать обучение), но, думаю, снесу в некст версии ноутбука..._

In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import ConcatDataset, DataLoader, Dataset


def find_project_root():
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "data" / "lstm" / "meta.json").exists():
            return path
    raise FileNotFoundError("Сначала запустите 06_LSTM_Data_Preparation.ipynb")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "lstm"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)

with open(DATA_DIR / "meta.json", encoding="utf-8") as f:
    META = json.load(f)

LABELED_CUTOFFS = META["labeled_cutoffs"]
INFERENCE_CUTOFF = META["inference_cutoff"]
BASE_SEQUENCE_FEATURES = META["base_sequence_features"]
CALENDAR_FEATURES = META["calendar_sequence_features"]
STATIC_FEATURES = META["static_features"]
STATIC_LOG_FEATURES = META["static_log_copy_features"]

BASE_INDEX = {name: i for i, name in enumerate(BASE_SEQUENCE_FEATURES)}
STATIC_LOG_INDICES = [STATIC_FEATURES.index(name) for name in STATIC_LOG_FEATURES]

HOLDOUT_TRAIN_CUTOFFS = LABELED_CUTOFFS[:-1]
HOLDOUT_CUTOFF = LABELED_CUTOFFS[-1]

BATCH_SIZE = 1024
CLASSIFIER_EPOCHS = 3
REGRESSOR_EPOCHS = 13
LR = 5e-4
WEIGHT_DECAY = 5e-4
RANDOM_STATE = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
NUM_WORKERS = 4 if DEVICE.type == "cuda" else 0


def set_seed():
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)


set_seed()
print("device:", DEVICE)
print("holdout:", HOLDOUT_CUTOFF)

device: mps
holdout: 2026-01-14


## Dataset

Для обеих моделей использую один и тот же объект: 90-дневную последовательность + static snapshot.  
Regressor просто получает только строки с `y > 0`.

In [2]:
class HybridDataset(Dataset):
    def __init__(self, cutoff, with_target=True, positive_only=False):
        path = DATA_DIR / cutoff
        self.X = np.load(path / "X.npy", mmap_mode="r")
        self.static = np.load(path / "static.npy", mmap_mode="r")
        self.calendar = np.load(path / "calendar.npy", mmap_mode="r").astype(np.float32)
        self.users = np.load(path / "user_id.npy", mmap_mode="r")
        self.y = np.load(path / "y.npy", mmap_mode="r") if with_target else None
        self.indices = np.flatnonzero(self.y > 0) if positive_only else None

    def __len__(self):
        return len(self.X) if self.indices is None else len(self.indices)

    def __getitem__(self, i):
        j = i if self.indices is None else int(self.indices[i])

        sequence = np.concatenate([
            np.asarray(self.X[j], dtype=np.float32),
            self.calendar,
        ], axis=1)
        static = np.array(self.static[j], dtype=np.float32)

        sequence = torch.from_numpy(sequence)
        static = torch.from_numpy(static)

        if self.y is None:
            return sequence, static

        y = torch.tensor(float(self.y[j]), dtype=torch.float32)
        return sequence, static, y


def make_loader(cutoffs, shuffle=False, with_target=True, positive_only=False):
    if isinstance(cutoffs, str):
        cutoffs = [cutoffs]

    dataset = ConcatDataset([
        HybridDataset(
            cutoff,
            with_target=with_target,
            positive_only=positive_only,
        )
        for cutoff in cutoffs
    ])

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
        drop_last=shuffle,
    )

## Динамические признаки

К 13 базовым каналам и 6 календарным добавляю те же 21 признака из пред-предыдущего коммита:

- события `has_*`;
- conversion ratios;
- rolling mean за 7/30 дней;
- rolling activity rate за 7/30 дней;
- первые разности.

Rolling считаю только по текущему и предыдущим дням. Все 90 дней находятся до target, поэтому BiLSTM не создает leakage (в целом все проверил, каж-ся норм).

In [3]:
def causal_mean(values, window):
    values = F.pad(values.unsqueeze(1), (window - 1, 0))
    return F.avg_pool1d(values, window, stride=1).squeeze(1)


def first_difference(values):
    return F.pad(values[:, 1:] - values[:, :-1], (1, 0))


def safe_ratio(numerator, denominator, max_value=5.0):
    ratio = numerator / denominator.clamp_min(1e-3)
    ratio = torch.where(denominator > 0, ratio, torch.zeros_like(ratio))
    return ratio.clamp(0, max_value)


def make_sequence_features(x):
    searches_log = x[..., BASE_INDEX["searches"]]
    search_to_cart_log = x[..., BASE_INDEX["search_to_cart"]]
    search_to_ord_log = x[..., BASE_INDEX["search_to_ord"]]
    cat_to_cart_log = x[..., BASE_INDEX["cat_to_cart"]]
    cat_to_ord_log = x[..., BASE_INDEX["cat_to_ord"]]
    to_cart_log = x[..., BASE_INDEX["to_cart"]]
    to_ord_log = x[..., BASE_INDEX["to_ord"]]
    gmv_search_log = x[..., BASE_INDEX["gmv_search"]]
    gmv_log = x[..., BASE_INDEX["gmv"]]
    active = x[..., BASE_INDEX["active"]]

    searches = torch.expm1(searches_log).clamp_min(0)
    search_to_cart = torch.expm1(search_to_cart_log).clamp_min(0)
    search_to_ord = torch.expm1(search_to_ord_log).clamp_min(0)
    cat_to_cart = torch.expm1(cat_to_cart_log).clamp_min(0)
    cat_to_ord = torch.expm1(cat_to_ord_log).clamp_min(0)
    to_cart = torch.expm1(to_cart_log).clamp_min(0)
    to_ord = torch.expm1(to_ord_log).clamp_min(0)
    gmv_search = torch.expm1(gmv_search_log).clamp_min(0)
    gmv = torch.expm1(gmv_log).clamp_min(0)

    derived = [
        (search_to_cart > 0).float(),
        (search_to_ord > 0).float(),
        (cat_to_cart > 0).float(),
        (cat_to_ord > 0).float(),
        safe_ratio(search_to_cart, searches),
        safe_ratio(search_to_ord, searches),
        safe_ratio(to_ord, to_cart),
        safe_ratio(gmv_search, gmv, 1.5),
    ]

    for values in [searches_log, to_cart_log, to_ord_log, gmv_log]:
        derived += [causal_mean(values, 7), causal_mean(values, 30)]

    derived += [
        causal_mean(active, 7),
        causal_mean(active, 30),
        first_difference(searches_log),
        first_difference(to_ord_log),
        first_difference(gmv_log),
    ]

    return torch.cat([x, torch.stack(derived, dim=-1)], dim=-1)


SEQ_INPUT_SIZE = len(BASE_SEQUENCE_FEATURES) + len(CALENDAR_FEATURES) + 21
print("sequence features:", SEQ_INPUT_SIZE)

sequence features: 40


## Static preprocessing

Использую все 95 static-признаков из [06_LSTM_Data_Preparation.ipynb](./06_LSTM_Data_Preparation.ipynb).

К ним добавляю:

- `log1p`-копии 51 heavy-tail признака;
- missing mask для всех 95 исходных признаков.

Получается 241 static-признак.

Среднее и стандартное отклонение всегда считаю только по train-cutoff. Для финального обучения train -- все размеченные cutoff; inference в статистику не попадает.

In [4]:
STATIC_INPUT_SIZE = 2 * len(STATIC_FEATURES) + len(STATIC_LOG_INDICES)


def augment_static_numpy(raw):
    raw = np.asarray(raw, dtype=np.float32)
    source = raw[:, STATIC_LOG_INDICES]

    logs = np.where(
        np.isfinite(source),
        np.log1p(np.clip(source, 0, None)),
        np.nan,
    ).astype(np.float32)

    missing = (~np.isfinite(raw)).astype(np.float32)
    return np.concatenate([raw, logs, missing], axis=1)


def fit_static_stats(cutoffs, chunk_size=65_536):
    sums = np.zeros(STATIC_INPUT_SIZE, dtype=np.float64)
    sums_sq = np.zeros(STATIC_INPUT_SIZE, dtype=np.float64)
    counts = np.zeros(STATIC_INPUT_SIZE, dtype=np.int64)

    for cutoff in cutoffs:
        raw = np.load(DATA_DIR / cutoff / "static.npy", mmap_mode="r")

        for start in range(0, len(raw), chunk_size):
            block = augment_static_numpy(raw[start:start + chunk_size])
            finite = np.isfinite(block)
            safe = np.where(finite, block, 0.0).astype(np.float64)

            sums += safe.sum(axis=0)
            sums_sq += (safe * safe).sum(axis=0)
            counts += finite.sum(axis=0)

    counts = np.maximum(counts, 1)
    mean = sums / counts
    std = np.sqrt(np.maximum(sums_sq / counts - mean * mean, 1e-6))

    return (
        torch.tensor(mean, dtype=torch.float32),
        torch.tensor(std, dtype=torch.float32),
    )


def normalize_static(raw, stats):
    source = raw[:, STATIC_LOG_INDICES]
    logs = torch.where(
        torch.isfinite(source),
        torch.log1p(source.clamp_min(0)),
        torch.nan,
    )

    augmented = torch.cat([
        raw,
        logs,
        (~torch.isfinite(raw)).float(),
    ], dim=1)

    mean, std = stats
    mean = mean.to(raw.device)
    std = std.to(raw.device)

    augmented = torch.where(torch.isfinite(augmented), augmented, mean)
    return (augmented - mean) / std


print("static features:", STATIC_INPUT_SIZE)

static features: 241


## Архитектура

Обе задачи используют архитектуру из коммита ранее _(MLP classifier su**s (уважаю правила Конкурса!))_.

Sequence-ветка:

```text
40 -> BatchNorm -> Linear(96) -> 2-layer BiLSTM(128)
   -> last hidden + mean pooling + max pooling
   -> 256
```

Static-ветка:

```text
241 -> BatchNorm -> 256 -> 128
```

После этого объединяю обе ветки и получаю один logit / одно значение log-GMV.

In [5]:
class HybridLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.sequence_bn = nn.BatchNorm1d(SEQ_INPUT_SIZE)
        self.sequence_projection = nn.Sequential(
            nn.Linear(SEQ_INPUT_SIZE, 96),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        self.lstm = nn.LSTM(
            input_size=96,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.25,
            bidirectional=True,
        )

        self.sequence_head = nn.Sequential(
            nn.LayerNorm(128 * 6),
            nn.Linear(128 * 6, 256),
            nn.GELU(),
            nn.Dropout(0.30),
        )

        self.static_head = nn.Sequential(
            nn.BatchNorm1d(STATIC_INPUT_SIZE),
            nn.Linear(STATIC_INPUT_SIZE, 256),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.30),
        )

        self.fusion = nn.Sequential(
            nn.Linear(384, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, 1),
        )

    def forward(self, sequence, static):
        sequence = make_sequence_features(sequence)
        sequence = self.sequence_bn(sequence.transpose(1, 2)).transpose(1, 2)
        sequence = self.sequence_projection(sequence)

        output, (hidden, _) = self.lstm(sequence)

        last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        sequence_repr = self.sequence_head(torch.cat([
            last_hidden,
            output.mean(dim=1),
            output.amax(dim=1),
        ], dim=1))

        static_repr = self.static_head(static)
        return self.fusion(torch.cat([sequence_repr, static_repr], dim=1)).squeeze(1)


probe = HybridLSTM()
print("parameters:", f"{sum(p.numel() for p in probe.parameters()):,}")
del probe

parameters: 1,040,019


## Обучение

Гиперпараметры уже выбраны предыдущим запуском, в т.ч. число эпох, поправлю...

Classifier использует BCE по `y > 0`.

Regressor видит только `y > 0` и минимизирует MSE для `log1p(y)`.

In [6]:
def train_one_epoch(model, loader, optimizer, task, static_stats):
    model.train()
    total_loss = 0.0
    total_n = 0

    for sequence, static, y in loader:
        sequence = sequence.to(DEVICE)
        static = normalize_static(static.to(DEVICE), static_stats)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        output = model(sequence, static)

        if task == "classifier":
            loss = F.binary_cross_entropy_with_logits(output, (y > 0).float())
        else:
            loss = F.mse_loss(output, torch.log1p(y))

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(y)
        total_n += len(y)

    return total_loss / total_n


def fit_model(task, cutoffs, static_stats, epochs):
    set_seed()

    loader = make_loader(
        cutoffs,
        shuffle=True,
        positive_only=task == "regressor",
    )

    model = HybridLSTM().to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=1e-6,
    )

    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, loader, optimizer, task, static_stats)
        scheduler.step()
        print(f"{task:10s} | epoch {epoch:02d}/{epochs:02d} | loss={loss:.5f}")

    return model

## Предсказание

Classifier возвращает logit, regressor -- conditional `log1p(GMV)`.

Склеиваю их **до `expm1`**:

```text
p = sigmoid(classifier_logit)
pred_log = p * regressor_log
prediction = expm1(pred_log)
```

Трешхолда тоже (пока) нет.

In [7]:
@torch.no_grad()
def predict_raw(model, cutoff, static_stats, with_target=True):
    dataset = HybridDataset(cutoff, with_target=with_target)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
    )

    model.eval()
    outputs = []
    targets = []

    for batch in loader:
        if with_target:
            sequence, static, y = batch
            targets.append(y.numpy())
        else:
            sequence, static = batch

        sequence = sequence.to(DEVICE)
        static = normalize_static(static.to(DEVICE), static_stats)
        outputs.append(model(sequence, static).cpu().numpy())

    y = np.concatenate(targets) if with_target else None
    return np.asarray(dataset.users), np.concatenate(outputs), y


def make_prediction(classifier_logits, regressor_log):
    probability = 1.0 / (1.0 + np.exp(-np.clip(classifier_logits, -30, 30)))
    pred_log = probability * np.clip(regressor_log, 0, None)
    return pred_log, np.expm1(pred_log)


def rmsle_from_log(y, pred_log):
    return float(np.sqrt(np.mean((pred_log - np.log1p(y)) ** 2)))

## Holdout

Настройки уже взяты из предыдущего эксперимента, поэтому могу честно обучиться на первых 9 cutoff и один раз посмотреть на последний.

Holdout нигде дальше не используется для выбора архитектуры, эпох или gate.

In [8]:
holdout_static_stats = fit_static_stats(HOLDOUT_TRAIN_CUTOFFS)

classifier = fit_model(
    "classifier",
    HOLDOUT_TRAIN_CUTOFFS,
    holdout_static_stats,
    CLASSIFIER_EPOCHS,
)

regressor = fit_model(
    "regressor",
    HOLDOUT_TRAIN_CUTOFFS,
    holdout_static_stats,
    REGRESSOR_EPOCHS,
)

_, holdout_logits, y_holdout = predict_raw(
    classifier,
    HOLDOUT_CUTOFF,
    holdout_static_stats,
)
_, holdout_reg_log, _ = predict_raw(
    regressor,
    HOLDOUT_CUTOFF,
    holdout_static_stats,
)

holdout_pred_log, holdout_pred = make_prediction(
    holdout_logits,
    holdout_reg_log,
)

print("holdout RMSLE:", f"{rmsle_from_log(y_holdout, holdout_pred_log):.6f}")
print("true zeros:", f"{(y_holdout == 0).mean():.2%}")
print("exact predicted zeros:", f"{(holdout_pred == 0).mean():.2%}")

classifier | epoch 01/03 | loss=0.47864
classifier | epoch 02/03 | loss=0.47621
classifier | epoch 03/03 | loss=0.47495
regressor  | epoch 01/13 | loss=1.62661
regressor  | epoch 02/13 | loss=1.32174
regressor  | epoch 03/13 | loss=1.30827
regressor  | epoch 04/13 | loss=1.30191
regressor  | epoch 05/13 | loss=1.29821
regressor  | epoch 06/13 | loss=1.29193
regressor  | epoch 07/13 | loss=1.28852
regressor  | epoch 08/13 | loss=1.28515
regressor  | epoch 09/13 | loss=1.28269
regressor  | epoch 10/13 | loss=1.28131
regressor  | epoch 11/13 | loss=1.27985
regressor  | epoch 12/13 | loss=1.27817
regressor  | epoch 13/13 | loss=1.27767
holdout RMSLE: 1.679061
true zeros: 45.93%
exact predicted zeros: 0.00%


> btw лучший лосс из всех моделей ранее...

## Финальное обучение

Теперь обучаю те же две модели на всех 10 размеченных cutoff, Inference-cutoff не участвует.

In [9]:
del classifier, regressor

final_static_stats = fit_static_stats(LABELED_CUTOFFS)

final_classifier = fit_model(
    "classifier",
    LABELED_CUTOFFS,
    final_static_stats,
    CLASSIFIER_EPOCHS,
)

final_regressor = fit_model(
    "regressor",
    LABELED_CUTOFFS,
    final_static_stats,
    REGRESSOR_EPOCHS,
)

classifier | epoch 01/03 | loss=0.47795
classifier | epoch 02/03 | loss=0.47547
classifier | epoch 03/03 | loss=0.47428
regressor  | epoch 01/13 | loss=1.59373
regressor  | epoch 02/13 | loss=1.31815
regressor  | epoch 03/13 | loss=1.30508
regressor  | epoch 04/13 | loss=1.29444
regressor  | epoch 05/13 | loss=1.28738
regressor  | epoch 06/13 | loss=1.28416
regressor  | epoch 07/13 | loss=1.28050
regressor  | epoch 08/13 | loss=1.27751
regressor  | epoch 09/13 | loss=1.27467
regressor  | epoch 10/13 | loss=1.27238
regressor  | epoch 11/13 | loss=1.27029
regressor  | epoch 12/13 | loss=1.26958
regressor  | epoch 13/13 | loss=1.26918


## Submission

Сохраняю сабмит: [lstm_fixed_hyperparameters.csv](../../submissions/lstm_fixed_hyperparameters.csv) **(переименовал!!!)**.

In [10]:
user_ids, inference_logits, _ = predict_raw(
    final_classifier,
    INFERENCE_CUTOFF,
    final_static_stats,
    with_target=False,
)
_, inference_reg_log, _ = predict_raw(
    final_regressor,
    INFERENCE_CUTOFF,
    final_static_stats,
    with_target=False,
)

_, pred = make_prediction(inference_logits, inference_reg_log)

submission = pd.read_csv(PROJECT_ROOT / "data" / "sample_submit.csv")
submission["predict"] = submission["user_id"].map(
    pd.Series(pred, index=user_ids)
).clip(lower=0)

out_path = SUBMISSION_DIR / "lstm.csv"
submission.to_csv(out_path, index=False)

print("saved:", out_path)
print("exact zeros:", f"{(submission['predict'] == 0).mean():.2%}")
submission.head()

saved: /Users/pinta/Dev/E-CUP-2026/submissions/lstm.csv
exact zeros: 0.00%


,user_id,predict
0,2,1.940560
1,7,83.956520
2,15,6.223347
3,18,129.363602
4,23,0.396807


> ноутбук еще обновлю в плане числа эпох и экспериментов с трешхолдом.  
> скор предикта -- 1.6529693117, аналогичен предикту того, на что все это время ссылался.